[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/purva-thakre/qce26-qrisp/blob/Pietro_notebooks_Colab_links/Session_2_Introduction_to_Jasp/JaspJIT.ipynb)

# JIT Compilation

We know how to get a `jaxpr` out of a function (and a `jaspr` from a qrisp quantum program). 

In this notebook we see what `jax.jit` actually does with it: how it turns that jaxpr into compiled code that runs fast.

Most of the concepts in this notebook (static arguments, control flow with JIT, etc.) are exactly the same concepts we have seen in the previous notebook. 

Here we just see the full pipeline instead of the first step (tracing).

## Setup

In [ ]:
!pip install qrisp

In [1]:
import jax
import jax.numpy as jnp
import time

## Applying `jax.jit` for the first time

The most natural way to perform Just-In-Time compilation with JAX is through `jax.jit`. Let's see it in action.

We define a plain `f` Python function, which does some repeated array math.

In [3]:
def f(x):
    for _ in range(50):
        x = 0.25 * x**3 + 0.75 * x
    return jnp.sum(x)

At first, we time it as-is, so we have a plain baseline to compare every compiled version against later.

Every timed call below ends in `.block_until_ready()`. This is necessary because JAX dispatches work asynchronously, so without it we'd be timing how fast Python hands off the call, not how long the computation actually takes (more on exactly why in the "Asynchronous Dispatch" section).

In [4]:
x = jnp.ones((1000, 1000))

start = time.perf_counter()
for _ in range(50):
    f(x).block_until_ready()
print(f"Without (jit): {(time.perf_counter() - start) / 50 * 1000:.2f} ms per call")

Without (jit): 120.46 ms per call


Let's see what changes once `jax.jit` gets involved.

`jax.jit(f)` doesn't compile anything by itself. It just wraps `f` so the *first* call traces and compiles it, and later calls (with matching input shapes/dtypes) **reuse** that compiled version.

Watch the timings below shift as we go from "no jit" to "first jit call" to "cached jit call."

In [5]:
f_jit = jax.jit(f)

In [6]:
# First call (tracing + compilation)
start = time.perf_counter()
for _ in range(50):
    f_jit(x).block_until_ready()
elapsed = (time.perf_counter() - start) / 50 * 1000
print(f"First call (trace + compile): {elapsed:.2f} ms per call")

First call (trace + compile): 6.61 ms per call


In [7]:
# Subsequent calls (use the cached compiled function)
start = time.perf_counter()
for _ in range(50):
    f_jit(x).block_until_ready()
print(f"Subsequent calls: {(time.perf_counter() - start) / 50 * 1000:.2f} ms per call")

Subsequent calls: 3.54 ms per call


Observations:

- If we maintain the same shape and type of dynamic arguments, then the jaxpr is the same and, therefore, JAX does not have to re-compile our function: we have true just-in-time compilation!

- In practice, JAX doesn't retrace and compare jaxprs to decide this: it keys its cache on the input shapes/dtypes directly, so a matching signature skips tracing entirely and reuses the cached compiled version; a new signature forces a fresh trace (and, since the computation may now look different, possibly a different jaxpr).

- The first call already blends one real compilation together with 49 cache hits in the same timed loop. That's why its average doesn't look as expensive as a raw compile would.

We can also write the same function using `@jax.jit` as a decorator instead of calling `jax.jit(f)` directly:

In [7]:
@jax.jit
def f_decorated(x):
    for _ in range(50):
        x = 0.25 * x**3 + 0.75 * x
    return jnp.sum(x)


_ = f_decorated(x).block_until_ready()

## Asynchronous Dispatch and `block_until_ready` (bonus)

Calling a jitted function hands work off to the accelerator and returns right away. It doesn't block until the result is actually computed.

The first call below also doubles as a warm-up (so the timings that follow aren't paying a compilation cost); `block_until_ready()` is what forces Python to actually wait for a result.

In [8]:
@jax.jit
def heavy(a):
    return a @ a


a = jnp.ones((2000, 2000))
# this call triggers trace + compile.
# That is, a warm-up before we time dispatch vs. compute separately
heavy(a).block_until_ready()

Array([[2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       ...,
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.]], dtype=float32)

In [9]:
start = time.perf_counter()
result = heavy(a)
print(f"Call returned after: {(time.perf_counter() - start) * 1000:.3f} ms")

Call returned after: 1.293 ms


In [10]:
start = time.perf_counter()
heavy(a).block_until_ready()
print(f"Total time (dispatch + compute): {(time.perf_counter() - start) * 1000:.3f} ms")

Total time (dispatch + compute): 40.563 ms


Observations:

- The bare call above returns in a fraction of a millisecond. This is dispatch overhead, not the matmul itself: the accelerator keeps working after Python already moved on.

- Adding `block_until_ready()` exposes the real total time, dispatch plus compute together. This is the async gap in action.

## The Compilation Cache (bonus)

The jit cache is keyed on the *shape and dtype* of the inputs, not their concrete values. Same shape means a cache hit, a new shape means JAX has never seen this "version" of the function before.

This is exactly the same behaviour we have already seen for the `jaxpr`, which is conceptually the first IR of the entire Just-In-Time compilation process.
The difference now is that `jit` doesn't stop at the `jaxpr`: it keeps compiling all the way down to machine code. We open up exactly what happens in between later in this notebook, in "Inspecting What Gets Compiled".

The three calls below go compile → cache-hit → recompile, back to back.

In [11]:
@jax.jit
def g(x):
    return x * 2.0


# Compile for shape (10,)
start = time.perf_counter()
_ = g(jnp.ones(10)).block_until_ready()
print(f"Compiled for shape (10,): {(time.perf_counter() - start) * 1000:.2f} ms")

# Cache hit (same shape)
start = time.perf_counter()
_ = g(jnp.ones(10)).block_until_ready()
print(f"Compiled for shape (10,): {(time.perf_counter() - start) * 1000:.2f} ms")

# Recompile (new shape)
start = time.perf_counter()
_ = g(jnp.ones(20)).block_until_ready()
print(f"Recompile for shape (20,): {(time.perf_counter() - start) * 1000:.2f} ms")

Compiled for shape (10,): 43.60 ms
Compiled for shape (10,): 0.53 ms
Recompile for shape (20,): 40.01 ms


Same shape → fast (cache hit). 

new shape → slow again (fresh trace + compile), because the cache has no way to reuse a compiled program for an input signature it hasn't seen.

## Boolean Simulation

We've just seen `jax.jit`'s cache decide, based only on shape and dtype, whether to reuse a compiled program or trace again. Now let's see the first way Jasp gets to reuse that exact same compiled pipeline.

If a whole circuit is built only out of classical gates (that is, gates such as X, CX, CCX, SWAP, etc. which only ever permute basis states), there is no need for a real quantum-state simulator at all. That is, every qubit is either `0` or `1`, always, and every gate is just an ordinary bit operation.

`boolean_simulation` is a decorator defined within the Jasp submodule which exploits exactly this. It rewrites the Jaspr into a version with no quantum primitives left at all (qubits become classical bits packed into an array and gates become bitwise operations on that array). And because it's now ordinary JAX code, it goes through the real `jax.jit` pipeline we have seen above.

The catch: it only works for circuits that never leave classical territory. Anything that creates superposition (`h`, `rx`, `ry`, ...) will raise an exception.

In [8]:
from qrisp import QuantumFloat, measure
from qrisp.jasp import boolean_simulation, jrange


@boolean_simulation
def main(i, j):
    a = QuantumFloat(10)
    b = QuantumFloat(10)
    a[:] = i
    b[:] = j

    c = QuantumFloat(20)
    for _ in jrange(150):
        c += a + b

    return measure(c)

/home/pietrop/anaconda3/envs/Qrisp_3_11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from qrisp import count_ops


# Same circuit body, but count_ops (not boolean_simulation) to see its actual size
@count_ops(meas_behavior="0")
def main_gate_count(i, j):
    a = QuantumFloat(10)
    b = QuantumFloat(10)
    a[:] = i
    b[:] = j

    c = QuantumFloat(20)
    for _ in jrange(150):
        c += a + b

    return measure(c)


gate_counts = main_gate_count(3, 4)
print(f"Total gates in this circuit: {sum(gate_counts.values())}")

# Initial run (compilation)
start = time.perf_counter()
result = main(3, 4)
print(f"Result: {result}, Time: {(time.perf_counter() - start) * 1000:.2f} ms")

# Subsequent run (cache hit)
start = time.perf_counter()
result = main(3, 4)
print(f"Result: {result}, Time: {(time.perf_counter() - start) * 1000:.2f} ms")

Observations:

- `main(3, 4)` adds `3 + 4` into `c`, 150 times: `1050 = (3+4) * 150`. Purely classical arithmetic and purely classical result.

- This example has almost 90,000 gates, and yet both calls finish in a fraction of a second. We can test examples at scale, at native JAX speed, with none of the exponential cost of a real quantum-state simulation.

### Catching Bad Uncomputation (bonus)

`boolean_simulation` also double-checks something for free: every deleted `QuantumVariable` must be back in the $\ket{0}$ state (properly uncomputed). Otherwise we are silently discarding information. Since every qubit is a real classical bit here, checking this is just checking whether that bit is `0`:

In [14]:
from qrisp import QuantumFloat
from qrisp.jasp import boolean_simulation


@boolean_simulation
def main(i):
    a = QuantumFloat(10)
    a[:] = i
    a.delete()
    return


main(0)
print("---")
main(1)
print("---")
main(3)

---
---


Observations:

- `main(0)`: `a` starts and stays at `0`, so deleting it is valid: no warning.

- `main(1)`: `a = 1` is `0000000001` in binary, one qubit is `1`: one warning.

- `main(3)`: `a = 3` is `0000000011` in binary, two qubits are `1`: two warnings.

## `qache`: Jasp's Compilation Cache

This is the second (and last) decorator that we are going to see in Jasp. 
It is the mechanism qrisp's Jasp module reuses for caching.

In Jasp, the `@qache` decorator allows us to mark a function as 'reusable' (that is, the jasp expression of this function will be cached and reused in the next calls if the function is called with the same signature).

Using the qache decorator not only improves the compilation speed but also enables the compiler to speed up transformation processes.

Below, we create a simple function that is 'qached'. 
To simulate an expensive compilation task we insert a `time.sleep` command:

In [12]:
from qrisp import h, cx, measure, QuantumVariable, QuantumFloat, make_jaspr
from qrisp.jasp import qache


@qache
def qached_function(qv):
    h(qv[0])
    cx(qv[0], qv[1])
    res_bl = measure(qv[0])
    # Simulate demanding compilation procedure by calling
    time.sleep(1)
    return res_bl


def main():
    a = QuantumVariable(2)
    b = QuantumFloat(2)
    bl_0 = qached_function(a)
    bl_1 = qached_function(b)
    bl_2 = qached_function(a)
    bl_3 = qached_function(b)

    return bl_0 & bl_1 & bl_2 & bl_3


# Measure the time required for tracing
start = time.perf_counter()
jaspr = make_jaspr(main)()
print(f"Tracing time: {(time.perf_counter() - start) * 1000:.2f} ms")

Tracing time: 2020.12 ms


Even though `qached_function` has been called 4 times, we only see a delay of 2 seconds. 

This is because the function has been called with two different quantum types, implying it has been traced twice and recalled from the cache twice. 

We can see all of this by looking at the `jaspr`:

In [13]:
print(jaspr)

let qached_function = { lambda ; a:QubitArray b:QuantumState. let
    c:Qubit = jasp.get_qubit a 0:i64[]
    d:QuantumState = jasp.quantum_gate[gate=h] c b
    e:Qubit = jasp.get_qubit a 1:i64[]
    f:QuantumState = jasp.quantum_gate[gate=cx] c e d
    g:bool[] h:QuantumState = jasp.measure c f
  in (g, h) } in
let qached_function1 = { lambda ; i:QubitArray j:i64[] k:QuantumState. let
    l:Qubit = jasp.get_qubit i 0:i64[]
    m:QuantumState = jasp.quantum_gate[gate=h] l k
    n:Qubit = jasp.get_qubit i 1:i64[]
    o:QuantumState = jasp.quantum_gate[gate=cx] l n m
    p:bool[] q:QuantumState = jasp.measure l o
  in (p, q) } in
{ lambda ; r:QuantumState. let
    s:QubitArray t:QuantumState = jasp.create_qubits 2:i64[] r
    u:QubitArray v:QuantumState = jasp.create_qubits 2:i64[] t
    w:bool[] x:QuantumState = jit[name=qached_function jaxpr=qached_function] s v
    y:bool[] z:QuantumState = jit[name=qached_function jaxpr=qached_function1] u
      0:i64[] x
    ba:bool[] bb:QuantumState

We see three different function definitions in the `jaspr`:

- The first one describes `qached_function` called with a `QuantumVariable`. For this kind of signature only the `QubitArray` is required.

- The second one describes `qached_function` called with `QuantumFloat`. Additionally to the `QubitArray`, the `.exponent` attribute is also passed as a traced value (the extra `i64[]` argument).

- The third function definition is `main`, which calls the previously defined functions.

Notice that a `jaspr` can be simulated using the built-in real-time simulator. We achieve this by calling the jaspr like a function:

In [17]:
jaspr()

Array(False, dtype=bool)

This is qrisp's own default simulator. That is, not the pipeline `boolean_simulation` used earlier in this notebook. 

Calling a jaspr this way still walks every quantum primitive through a custom Python interpreter, one equation at a time (never through `jit` → StableHLO → XLA). `boolean_simulation` only reaches that pipeline because it removes every quantum primitive first. By default, Jasp does not.

## Static Arguments (bonus)

Sometimes an argument controls the *structure* of the computation itself. Here `n` decides how many times we loop, so it has to be fixed at trace time, not just known at runtime.

`static_argnums=(1,)` tells `jit` to treat that argument as a compile-time constant, tracing a fresh version of the function for every distinct value it sees.

This is exactly the same concept we have seen in the previous notebook. 
Here we see how providing a different **value** for a static argument determines a full recompilation process.

In [18]:
from functools import partial


# static_argnums=(1,): n becomes a compile-time constant, not a traced value
@partial(jax.jit, static_argnums=(1,))
def apply_n_times(x, n):
    for _ in range(n):
        x = jnp.tanh(x)
    return x

In [19]:
x = jnp.ones(4)

Let's call it with `n=3` twice, then `n=5`, and watch for the same compile → cache-hit → recompile pattern as before:

In [20]:
start = time.perf_counter()
_ = apply_n_times(x, 3).block_until_ready()
print(f"Compiles for n=3: {(time.perf_counter() - start) * 1000:.2f} ms")

start = time.perf_counter()
_ = apply_n_times(x, 3).block_until_ready()
print(f"Cache hit: {(time.perf_counter() - start) * 1000:.2f} ms")

start = time.perf_counter()
_ = apply_n_times(x, 5).block_until_ready()
print(f"Recompiles for n=5: {(time.perf_counter() - start) * 1000:.2f} ms")

Compiles for n=3: 60.05 ms
Cache hit: 1.65 ms
Recompiles for n=5: 57.12 ms


Same pattern as the shape-keyed cache above, just keyed on `n`'s static value instead of a shape: same `n` → cache hit; different `n` → a full retrace, because with `n` static, changing it really does produce a different jaxpr (a different number of unrolled `tanh` calls baked in).

## Pitfall: Side Effects Inside jit (bonus)

As with tracing in general, a plain Python `print` only fires while JAX is tracing the function to build its jaxpr, not on every call.

Watch which of the three calls below actually print.

In [21]:
@jax.jit
def f_with_print(x):
    # plain Python print, not jax.debug.print
    print("Python print - fires at trace time only!")
    return x + 1.0

In [22]:
print("=== Call 1 ===")
_ = f_with_print(jnp.array(1.0)).block_until_ready()

print("=== Call 2 ===")
_ = f_with_print(jnp.array(2.0)).block_until_ready()

print("=== Call 3 (new shape - recompiles) ===")
_ = f_with_print(jnp.ones(3)).block_until_ready()

=== Call 1 ===
Python print - fires at trace time only!
=== Call 2 ===
=== Call 3 (new shape - recompiles) ===
Python print - fires at trace time only!


Observations:

- Call 1 and Call 3 both trace (call 3 because its shape is new), so both print.

- Call 2 is a cache hit (same shape as call 1): no retrace, no print.

Let's contrast that with `jax.debug.print`, built specifically to print real values from inside compiled code:

In [23]:
@jax.jit
def f_with_debug_print(x):
    # jax.debug.print (not Python's print) runs at actual call time, inside the compiled program
    jax.debug.print("Value of x at runtime: {}", x)
    return x + 1.0


# Uncomment to explore: with @jax.jit removed, the jaxpr shows debug_callback directly,
# instead of wrapped in a nested `jit` (nested jit is covered in its own section below)
# jax.make_jaxpr(f_with_debug_print)(jnp.ones(3))

`jax.debug.print` is different: it compiles down to a real primitive (`debug_callback`, carrying a `Debug` effect) baked into the program itself, so it prints the real runtime value on every call, not just at trace time.

In [24]:
_ = f_with_debug_print(jnp.ones(3)).block_until_ready()
_ = f_with_debug_print(jnp.array(2.0)).block_until_ready()

Value of x at runtime: [1. 1. 1.]
Value of x at runtime: 2.0


Both calls print here, with the actual runtime values. 

Unlike the Python `print` above, this one survives compilation instead of only firing while tracing.

## Python Control Flow on Dynamic Values Inside jit (bonus)

This is the same underlying problem as `broken_function` when we traced it directly with `jax.make_jaxpr` earlier: `jit` traces its function the same way, so a tracer still can't answer a Python `if`.

We have seen that, inside a jitted function `x` is a tracer, not a concrete number, and Python's `if` needs a real `bool`, so branching on a traced value can't work the way plain Python expects.

In [25]:
@jax.jit
def broken_function(x):
    # x is a tracer here - Python's "if" needs a concrete bool, which a tracer can't give
    if x > 0:
        return x * 2.0
    else:
        return x

Calling it hits exactly that wall:

In [26]:
try:
    broken_function(jnp.array(1.0))
except Exception as e:
    print(type(e).__name__)
    print(e)

TracerBoolConversionError
Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function broken_function at /tmp/ipykernel_100783/1365103922.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument x.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError


`TracerBoolConversionError` is JAX telling you it tried to turn a traced array into a Python `bool` for the `if`. This is the same static/dynamic distinction from jaxprs, only this time a placeholder can't just flow through: Python's `if` needs a concrete answer right now, so JAX raises a hard error instead.

Same fix as in the previous notebook. We can use `jax.lax.cond`, which traces both branches into the jaxpr and lets the runtime predicate pick between them: no Python-level branching on a tracer required.

In [27]:
@jax.jit
def fixed_function(x):
    # lax.cond(pred, true_fn, false_fn, operand): both branches get traced, the runtime predicate picks the result
    return jax.lax.cond(x > 0, lambda x: x * 2.0, lambda x: x, x)

Let's call it with a positive and a negative input, to see each branch actually taken:

In [28]:
fixed_function(jnp.array(1.0))

Array(2., dtype=float64, weak_type=True)

In [29]:
fixed_function(jnp.array(-1.0))

Array(-1., dtype=float64, weak_type=True)

The positive input takes the `x * 2.0` branch; the negative input takes the identity branch. The same two branches we saw traced into `cond` above, just picked at runtime by the predicate's concrete value.

### Dynamic Loops in Jasp: `jrange` (bonus)

The `if`-on-a-tracer pitfall above has a loop-shaped sibling: a plain Python `for i in range(n)` needs `n` to be a concrete Python integer at trace time, exactly like `if` needed a concrete `bool`. 

Jasp's `jrange` is the fix: the direct analogue of `jax.lax.fori_loop` from the previous notebook, spelled as an ordinary Python `for` loop instead of an explicit primitive call.

In [30]:
from qrisp import QuantumFloat, x, measure, control
from qrisp.jasp import jrange, make_jaspr, qache


@qache
def int_encoder(qv, encoding_int):
    for i in jrange(qv.size):
        with control(encoding_int & (1 << i)):
            x(qv[i])


def encode_test(a, b):
    qv = QuantumFloat(a)
    int_encoder(qv, b + 1)
    return measure(qv)


jaspr = make_jaspr(encode_test)(1, 1)
print(f"jaspr(5, 8): {jaspr(5, 8)}")
print(f"jaspr(5, 9): {jaspr(5, 9)}")

jaspr(5, 8): 9.0                                                                
jaspr(5, 9): 10.0                                                               


Observations:

- `jrange(qv.size)` looks exactly like a plain Python `for` loop, but `qv.size` can be a traced value. Jasp records one loop body plus a runtime bound, the same `while`-style structure `jax.lax.fori_loop` produced earlier, instead of unrolling.

- Encoding `b+1=9` into a 5-qubit register gives `9`. Encoding `10` gives `10`, which is a different *value* for the bound (no retrace) same as any other dynamic argument.

- Unlike `jax.lax.fori_loop`, a `jrange` loop can't carry a value out to the code after it. Trying to return a running total from inside the loop raises an exception (values from the *previous* iteration are fine, only reaching past the loop's own boundary isn't).

## Inspecting What Gets Compiled (bonus)

`jit` builds a compiled program behind the scenes, and `.lower(x)` lets you grab a handle on it without running it, and print it out at different levels of representation.

We have to make a distinction here: StableHLO is a portable, versioned dialect meant to be exchanged across tools and JAX/XLA versions (hence "stable"), while HLO is XLA's own internal representation, free to change under the hood. Below is the same function shown as both.

In [31]:
@jax.jit
def h(x):
    return jnp.sin(x) + jnp.cos(x)

In [32]:
x = jnp.ones(4)

In [33]:
lowered = h.lower(x)
print(lowered.as_text(dialect="stablehlo"))

module @jit_h attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<4xf64>) -> (tensor<4xf64> {jax.result_info = "result"}) {
    %0 = stablehlo.sine %arg0 : tensor<4xf64>
    %1 = stablehlo.cosine %arg0 : tensor<4xf64>
    %2 = stablehlo.add %0, %1 : tensor<4xf64>
    return %2 : tensor<4xf64>
  }
}



In [34]:
print(lowered.as_text(dialect="hlo"))

HloModule jit_h, entry_computation_layout={(f64[4]{0})->f64[4]{0}}

ENTRY main.5 {
  x.1 = f64[4]{0} parameter(0)
  sin.2 = f64[4]{0} sine(x.1)
  cos.3 = f64[4]{0} cosine(x.1)
  ROOT add.4 = f64[4]{0} add(sin.2, cos.3)
}




Same three operations either way. Notice HLO is basically the StableHLO version respelled in XLA's native syntax, not a different computation.

## `jit` Itself Is a Primitive (bonus)

As we saw with jaxprs, every JAX operation becomes a primitive in the trace, and `jit` is no exception.

When a jitted function is called from inside another traced function, it isn't inlined. It shows up as its own `jit` primitive wrapping a nested jaxpr.

In [35]:
def outer(arg):
    @jax.jit
    def inner(x):  # inner is jitted; outer itself stays a plain Python function
        return x + arg * jnp.ones(1)

    return arg + inner(arg - 2.0)

In [36]:
print(jax.make_jaxpr(outer)(1.0))

{ lambda ; a:f64[]. let
    b:f64[] = sub a 2.0:f64[]
    c:f64[1] = jit[
      name=inner
      jaxpr={ lambda ; a:f64[] b:f64[]. let
          d:f64[1] = broadcast_in_dim[
            broadcast_dimensions=()
            shape=(1,)
            sharding=None
          ] 1.0:f64[]
          e:f64[] = convert_element_type[new_dtype=float64 weak_type=False] a
          f:f64[1] = mul e d
          g:f64[] = convert_element_type[new_dtype=float64 weak_type=False] b
          c:f64[1] = add g f
        in (c,) }
    ] a b
    h:f64[] = convert_element_type[new_dtype=float64 weak_type=False] a
    i:f64[1] = add h c
  in (i,) }


See the `jit[name=inner, jaxpr={...}]` entry in the middle of the printout: that's `inner` kept as its own nested sub-program rather than flattened into `outer`'s jaxpr.

## `disable_jit` for Debugging (bonus)

`jax.disable_jit()` turns every `jit` inside the block into a no-op, so code runs as plain, eager Python: handy for stepping through with a debugger or trusting every `print` you see.

Compare how many times "Tracing!" prints with and without it:

In [37]:
@jax.jit
def f(x):  # reusing the name f for a fresh, simple demo - unrelated to the f from Setup
    print("Tracing!")
    return x * 2.0

In [38]:
print("=== With jit ===")
_ = f(jnp.array(1.0))
_ = f(jnp.array(2.0))

=== With jit ===
Tracing!


In [39]:
print("=== With disable_jit ===")
with jax.disable_jit():
    _ = f(jnp.array(1.0))
    _ = f(jnp.array(2.0))

=== With disable_jit ===
Tracing!
Tracing!


Observations:

- With `jit`: tracing (and the print) happens once. The second call is a cache hit and stays silent.

- With `disable_jit`: there's no cache at all, so every call re-runs the Python body from scratch, printing every time.

## Recompilation Diagnosis (bonus)

Unlike the static-argument example earlier, `n` here is just a regular (dynamic) argument: changing its *value* shouldn't force a retrace, only a change in shape/dtype would.

`trace_count` gives us a direct way to see how many times the function body actually gets traced, regardless of how many times it's called.

In [40]:
trace_count = [0]


@jax.jit
def counted(x, n):  # n is a regular (dynamic) argument here, not static
    trace_count[0] += 1
    return x * n

In [41]:
x = jnp.ones(4)
_ = counted(x, 3)
_ = counted(x, 3)
_ = counted(x, 5)
_ = counted(x, 3)

In [42]:
print(f"Number of traces: {trace_count[0]}")

Number of traces: 1


One trace total, even though `n` went from 3 to 5, because `n` isn't static, only the abstract shape/dtype of the arguments matters for the cache, not the concrete value.

Now the opposite scenario: what if we accidentally build a fresh jitted function on every call, instead of reusing one?

In [43]:
trace_count = [0]


def make_step():
    def step(x):
        trace_count[0] += 1
        return x * 2.0

    return jax.jit(
        step
    )  # a fresh function object wrapped in a fresh jax.jit, every time this is called

This time `make_step()` builds a brand-new Python closure (and wraps it in a brand-new `jax.jit`) on every call, even though the input shape never changes.

In [44]:
x = jnp.ones(4)
for _ in range(3):
    step = make_step()
    _ = step(x).block_until_ready()
print(f"Number of traces: {trace_count[0]}")

Number of traces: 3


We have three calls and three traces. JAX's cache is tied to the identity of the jitted function object itself, so a fresh closure every loop iteration means a "never seen before" function each time, and the cache never gets a chance to help.

## Converting to Catalyst dialect (bonus)

So far every jaxpr and jaspr in this notebook has run through either JAX's own `jit`/XLA path or Jasp's own Python-level interpreter. There's a third option: compiling all the way down to a real runtime via [Catalyst](https://docs.pennylane.ai/projects/catalyst/), which is Xanadu's own JIT/AOT compiler for JAX and PennyLane programs.

Catalyst is an optional extra, not installed by `qrisp` by default: `pip install qrisp[catalyst]`.

qrisp's `qjit` bridges *into* Catalyst by tracing our function to a `jaspr`, translating every Jasp primitive into Catalyst's own equivalent (`quantum_gate` → Catalyst's `qinst`, `measure` → Catalyst's own `measure`, and so on), then constructing and running an actual `catalyst.QJIT` instance. This is the exact same class a standalone PennyLane script would get from `@catalyst.qjit`. Just like `sin` lowers into `stablehlo.sine`, `qinst` itself lowers further into the `quantum.custom` ops shown below.

Its calling convention mirrors `jax.jit` exactly. The first call traces and compiles (now all the way to a runnable artifact), a later call with a matching signature is cached:

In [45]:
from qrisp import QuantumFloat, cx, h, measure, invert
from qrisp.jasp import qjit


@qjit
def test_fun(i):
    qv = QuantumFloat(i, -2)
    with invert():
        cx(qv[0], qv[qv.size - 1])
        h(qv[0])
    meas_res = measure(qv)
    return meas_res + 3


start = time.perf_counter()
result_first = test_fun(4)
elapsed = (time.perf_counter() - start) * 1000
print(f"First call (compile + run): {elapsed:.1f} ms -> {result_first}")

start = time.perf_counter()
result_cached = test_fun(4)
print(f"Cached call: {(time.perf_counter() - start) * 1000:.1f} ms -> {result_cached}")

First call (compile + run): 1070.3 ms -> 5.25
Cached call: 26.7 ms -> 3.0


Observations:

- This has the same shape as every other cache-hit demo in this notebook: a slow first call (trace + compile + run), a fast cached call. Just with a full hardware-compilation backend behind it instead of XLA.

- The two calls return different numbers even for the same `i=4`: `test_fun` genuinely measures a quantum state partway through, so its result is a real random variable, not a deterministic function of `i`.

See the translation directly: `.to_catalyst_mlir()` on a `jaspr` returns Catalyst's own MLIR dialect.

In [46]:
from qrisp import QuantumFloat, cx, h, measure
from qrisp.jasp import make_jaspr


def example_function(i):
    qv = QuantumFloat(i)
    cx(qv[0], qv[1])
    h(qv[1])
    meas_res = measure(qv)
    return meas_res + 1


jaspr = make_jaspr(example_function)(2)
print(jaspr.to_catalyst_mlir())

module @jaspr_function {
  func.func public @jit_jaspr_function(%arg0: tensor<i64>) -> tensor<f64> attributes {llvm.emit_c_interface} {
    %cst = stablehlo.constant dense<1.000000e+00> : tensor<f64>
    %c = stablehlo.constant dense<1024> : tensor<i64>
    %c_0 = stablehlo.constant dense<false> : tensor<i1>
    %c_1 = stablehlo.constant dense<1> : tensor<i64>
    %c_2 = stablehlo.constant dense<30> : tensor<i64>
    %c_3 = stablehlo.constant dense<[30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]> : tensor<30xi64>
    %c_4 = stablehlo.constant dense<0> : tensor<i64>
    %c0_i64 = arith.constant 0 : i64
    quantum.device shots(%c0_i64) ["/home/pietrop/anaconda3/envs/Qrisp_3_11/lib/python3.11/site-packages/pennylane_lightning/liblightning_qubit_catalyst.so", "LightningSimulator", "{'mcmc': False, 'num_burnin': 0, 'kernel_name': None}"] {auto_qubit_management}
    %0 = quantum.alloc( 20) : !quantum.reg
    %1 = stablehlo.broa

Observations:

- `quantum.device`, `quantum.alloc`, `quantum.custom "cx"` and so on are Catalyst's own dialect. This is a different spellings entirely from the `jasp.*` primitives used everywhere else in this notebook.

- Classical structure doesn't get reinvented for Catalyst: the `stablehlo.constant`/`stablehlo.while` machinery is still plain JAX StableHLO. Only the quantum-specific parts get translated into Catalyst's vocabulary.

## Keep Exploring

We've only had time to cover `qache`, `boolean_simulation`, and a first look at resource estimation and tracing. Jasp has quite a bit more: repeat-until-success loops, real-time classical control flow mixed with quantum operations (even running neural network inference inside a coherence window), and more.

For the full picture, see the [Jasp documentation](https://www.qrisp.eu/general/tutorial/Jasp.html). Hopefully this tutorial gave you a solid enough high-level understanding to keep exploring from here on your own.

## ... what comes next?

If you want to go further, the `BonusMaterial/` folder has three optional notebooks: 

- `PrimitivesInJax.ipynb` (build a custom JAX primitive from scratch, with rules for implementation, abstract evaluation, lowering, etc.),

- `XLABackend.ipynb` (how XLA actually compiles and fuses a jitted program), and 

- `StableHLOExport.ipynb` (inspecting compiled cost/memory, and exporting a compiled artifact).